# NYC Capital Projects Database (CPDB) — Exploration

Source: `Capital_Projects_Database_(CPDB)_-_Projects_(Points)` (NYC Dept. of City Planning).
One row = one capital project, with a point `geometry` and dollar amounts broken out by funding category.

**Column decoder** — the cryptic amount columns combine a *plan-stage prefix* with a *funding-category suffix*:

Prefixes: `pc` = planned commitment, `ad` = adopted, `al` = allocated, `co` = committed, `sp` = spent.

Suffixes: `ccnonex` = city cost non-exempt, `ccex` = city cost exempt, `cc` = total city cost, `nccstate`/`nccfed`/`nccother` = non-city cost by source, `ncc` = total non-city cost, `total` = grand total. `sptotalcb` = spent total, community-board level.

(If your copy of the dictionary disagrees on a prefix, trust the dictionary — the suffix structure is the reliable part.)

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Raw CSVs live in the repo's data/ folder (gitignored). Resolve whether the
# notebook kernel runs from the repo root or from notebooks/.
DATA = "data" if os.path.isdir("data") else "../data"
CSV = f"{DATA}/Capital_Projects_Database_(CPDB)_-_Projects_(Points)_20260624.csv"

In [ ]:
df = pd.read_csv(CSV, dtype=str, na_values=[""])
print(df.shape)
df.head()

In [ ]:
# Identify amount columns: every column whose name ends in a known funding suffix.
suffixes = ("ccnonex", "ccex", "cc", "nccstate", "nccfed", "nccother", "ncc", "total", "totalcb")
id_cols = ["ccpversion", "maprojid", "magenacro", "magency", "magenname",
           "descript", "projectid", "mindate", "maxdate", "typecat", "geometry"]
amount_cols = [c for c in df.columns if c not in id_cols]

# Strip thousands separators and convert to float
for c in amount_cols:
    df[c] = df[c].str.replace(",", "", regex=False).astype(float)

df["mindate"] = pd.to_datetime(df["mindate"], errors="coerce")
df["maxdate"] = pd.to_datetime(df["maxdate"], errors="coerce")
print(f"{len(amount_cols)} amount columns parsed")
df[amount_cols].describe().T.head(20)

In [ ]:
# Parse the MULTIPOINT geometry into lon/lat numeric columns
coord_re = re.compile(r"\(\(\s*(-?\d+\.?\d*)\s+(-?\d+\.?\d*)")

def parse_point(g):
    if not isinstance(g, str):
        return (np.nan, np.nan)
    m = coord_re.search(g)
    return (float(m.group(1)), float(m.group(2))) if m else (np.nan, np.nan)

df[["lon", "lat"]] = df["geometry"].apply(lambda g: pd.Series(parse_point(g)))
df[["geometry", "lon", "lat"]].head()

In [ ]:
# Column overview
overview = pd.DataFrame({
    "dtype": df.dtypes,
    "non_null": df.notna().sum(),
    "nunique": df.nunique(dropna=True),
})
overview

In [ ]:
# How many projects per managing agency, and total planned dollars (pctotal)
by_agency = (df.groupby("magenname")
               .agg(projects=("maprojid", "nunique"),
                    pctotal=("pctotal", "sum"))
               .sort_values("pctotal", ascending=False))
by_agency.head(20)

In [ ]:
# Project type categories
df["typecat"].value_counts(dropna=False)

In [ ]:
# Spend vs. plan: how far along are projects? (committed / planned)
tot = df[["pctotal", "adtotal", "altotal", "cototal", "sptotal"]].sum()
tot.plot(kind="bar", figsize=(8, 4),
         title="Citywide totals by plan stage (planned -> spent)")
plt.ylabel("$")
plt.tight_layout()
plt.show()
tot

In [ ]:
# Top 20 projects by planned total
(df[["maprojid", "magenname", "descript", "pctotal", "sptotal"]]
    .sort_values("pctotal", ascending=False)
    .head(20))

In [ ]:
# City vs. non-city funding mix (planned)
mix = pd.Series({
    "City cost": df["pccc"].sum(),
    "State": df["pcnccstate"].sum(),
    "Federal": df["pcnccfed"].sum(),
    "Other": df["pcnccother"].sum(),
})
mix.plot(kind="pie", autopct="%1.1f%%", figsize=(6, 6),
         title="Planned funding by source", ylabel="")
plt.tight_layout()
plt.show()
mix

In [ ]:
# Map: project locations, sized by planned total. Drop rows missing coords.
geo = df.dropna(subset=["lon", "lat"]).copy()
# Clip to NYC bounding box to drop any bad coordinates
geo = geo[(geo["lon"].between(-74.3, -73.6)) & (geo["lat"].between(40.45, 40.95))]

size = (geo["pctotal"].clip(lower=0).fillna(0)) ** 0.5
size = (size / size.max() * 200).clip(lower=3)

fig, ax = plt.subplots(figsize=(9, 9))
sc = ax.scatter(geo["lon"], geo["lat"], s=size, alpha=0.4,
                c=np.log1p(geo["pctotal"].clip(lower=0)), cmap="viridis")
ax.set_title("Capital projects (size & color ~ planned total)")
ax.set_xlabel("lon"); ax.set_ylabel("lat")
ax.set_aspect(1.3)
plt.colorbar(sc, label="log(1+pctotal)")
plt.tight_layout()
plt.show()

## Optional: geospatial joins / basemap
If you install `geopandas` + `contextily` you can overlay a real basemap and join to borough/community-district shapes:
```python
import geopandas as gpd, contextily as cx
gdf = gpd.GeoDataFrame(geo, geometry=gpd.points_from_xy(geo.lon, geo.lat), crs=4326).to_crs(3857)
ax = gdf.plot(markersize=size, alpha=0.5, figsize=(10,10))
cx.add_basemap(ax)
```
(Both are on conda-forge: `conda install -n nyc_climate geopandas contextily`.)

## Scratch